# REST API: поиск фильма через OMDb API
В этой версии API-ключ загружается из файла `.env`, а запрос и ответ проверяются на ошибки.

In [ ]:
import os
import json
import requests
from dotenv import load_dotenv

load_dotenv()

OMDB_URL = "https://www.omdbapi.com/"

In [ ]:
def get_api_key():
    key = os.getenv("OMDB_API_KEY")
    if not key:
        raise ValueError("Не найден OMDB_API_KEY. Создайте .env по образцу .env.example.")
    return key

def fetch_movie(title, api_key):
    response = requests.get(
        OMDB_URL,
        params={"t": title, "apikey": api_key},
        timeout=10,
    )
    response.raise_for_status()
    return response.json()

def print_movie(data):
    print(f"Название: {data.get('Title')}")
    print(f"Год: {data.get('Year')}")
    print(f"Жанр: {data.get('Genre')}")
    print(f"Режиссёр: {data.get('Director')}")
    print(f"Актёры: {data.get('Actors')}")
    print(f"Рейтинг IMDb: {data.get('imdbRating')}")

In [ ]:
title = input("Введите фильм для поиска: ").strip()

if not title:
    print("Название фильма не может быть пустым.")
else:
    try:
        json_data = fetch_movie(title, get_api_key())
    except ValueError as error:
        print(error)
    except requests.exceptions.RequestException as error:
        print("Ошибка сети или сервиса:", error)
    else:
        if json_data.get("Response") == "False":
            error_message = json_data.get("Error", "Неизвестная ошибка OMDb")
            if "invalid api key" in error_message.lower():
                print("OMDb отклонил API-ключ. Проверьте значение OMDB_API_KEY в .env.")
            else:
                print("Фильм не найден:", error_message)
        else:
            print(json.dumps(json_data, ensure_ascii=False, indent=2))
            print()
            print_movie(json_data)